# Task 3.2 — Failure Mode Analysis

## Failure Scenario: Over-segmentation with High DP Concentration and Low Margin Cost

**Scenario description:** I construct a dataset of two well-separated, compact, linearly-separable Gaussian blobs and configure iSVM with a high DP concentration parameter (α = 10) and a very low SVM cost (C1 = 0.1). I expect the method to fail here because:

1. **High α** (concentration parameter) encourages the DP to create many active components instead of concentrating on a few (the expected number of non-empty clusters grows as O(α log n) for a DP). With T=8 and α=10, the DP prior weakly penalises component proliferation.
2. **Low C1** (SVM margin-slack tradeoff) produces wide-margin, high-bias classifiers. When combined with small component membership weights (because the data is split across many components), the cost-sensitive SVM barely has enough "effective" data per component to find a meaningful separator.

This directly connects to **Assumption 3** from Task 1.2: the hinge-loss upper bound is tight only when there is sufficient data and a clear margin in each component. With many small components, the margin becomes trivial or infeasible, and the overall prediction degrades.


In [1]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

exec(open('/home/claude/isvm_core.py').read())

# Construct failure dataset: two very clean Gaussian blobs
X_fail, y_fail = make_blobs(n_samples=300, centers=2, cluster_std=0.5, random_state=42)
X_fail = StandardScaler().fit_transform(X_fail)
X_ftrain, X_ftest, y_ftrain, y_ftest = train_test_split(
    X_fail, y_fail, test_size=0.3, random_state=42, stratify=y_fail)

print(f"Failure dataset: {X_ftrain.shape[0]} train, {X_ftest.shape[0]} test")
print(f"Two linearly-separable Gaussian blobs (cluster_std=0.5)")


Failure dataset: 210 train, 90 test
Two linearly-separable Gaussian blobs (cluster_std=0.5)


In [1]:
# Global RBF-SVM — should work perfectly on this easy data
rbf_baseline = SVC(C=1.0, kernel='rbf', gamma='scale')
rbf_baseline.fit(X_ftrain, y_ftrain)
base_acc = accuracy_score(y_ftest, rbf_baseline.predict(X_ftest))
print(f"Global RBF-SVM (baseline) | Acc={base_acc:.4f}  (expected: ~1.0)")

# iSVM with high alpha and low C1 — failure mode
isvm_fail = iSVM(T=8, C1=0.1, C2=64.0, alpha=10.0, kernel='rbf', max_iter=30)
isvm_fail.fit(X_ftrain, y_ftrain)
fail_acc = isvm_fail.score(X_ftest, y_ftest)
fail_f1  = f1_score(y_ftest, isvm_fail.predict(X_ftest))
print(f"RBF-iSVM (α=10, T=8, C1=0.1) | Acc={fail_acc:.4f}  (FAILURE)")
print(f"  Active components: {isvm_fail.active_components()}")
print(f"  Accuracy drop vs baseline: {base_acc - fail_acc:.4f}")


Global RBF-SVM (baseline) | Acc=1.0000  (expected: ~1.0)
RBF-iSVM (α=10, T=8, C1=0.1) | Acc=0.4667  (FAILURE)
  Active components: 6
  Accuracy drop vs baseline: 0.5333


In [1]:
def plot_db(ax, predict_fn, X, y, title):
    h = 0.06
    x0,x1 = X[:,0].min()-0.5, X[:,0].max()+0.5
    y0,y1 = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx,yy = np.meshgrid(np.arange(x0,x1,h), np.arange(y0,y1,h))
    Z = predict_fn(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx,yy,Z,alpha=0.3,cmap=plt.cm.RdYlBu)
    ax.contour(xx,yy,Z,colors='black',linewidths=1.5)
    for cls,col,mk in zip([0,1],['#e74c3c','#3498db'],['s','*']):
        m=y==cls
        ax.scatter(X[m,0],X[m,1],c=col,marker=mk,s=50,zorder=5,edgecolors='k',linewidths=0.4,label=f'Class {cls}')
    ax.set_title(title, fontsize=10, fontweight='bold')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for cls,col,mk in zip([0,1],['#e74c3c','#3498db'],['s','*']):
    m=y_ftrain==cls
    axes[0].scatter(X_ftrain[m,0],X_ftrain[m,1],c=col,marker=mk,s=50,edgecolors='k',linewidths=0.3,label=f'Class {cls}')
axes[0].set_title('Failure Dataset\n(Clean Gaussian Blobs)', fontweight='bold')
axes[0].legend()
plot_db(axes[1], rbf_baseline.predict, X_ftrain, y_ftrain, f'Global RBF-SVM\nAcc={base_acc:.3f} ✓')
plot_db(axes[2], isvm_fail.predict, X_ftrain, y_ftrain,
        f'RBF-iSVM (α=10, T=8, C1=0.1)\nAcc={fail_acc:.3f} ✗ FAILS')
plt.suptitle('Failure Mode: Over-segmentation with High α and Low C1', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/home/claude/partB/results/failure_mode.png', dpi=150, bbox_inches='tight')
print("Saved failure_mode.png")


Saved failure_mode.png


## Explanation of Failure

The method fails catastrophically (accuracy ~47%, essentially random) on this simple dataset under high α and low C1. The failure mechanism is as follows: with α = 10 and T = 8, the DP stick-breaking prior distributes the 210 training samples across approximately 6 active components, giving each component roughly 35 effective samples (with fractional weights from ϕ). When C1 = 0.1, the cost of margin violations is very low, so the SVM in each component fits a very wide, imprecise margin — it essentially does not discriminate between classes within its cluster. At test time, a sample is assigned to the component whose Gaussian feature distribution it most resembles (not the component that classifies it correctly), and that component's low-C SVM predicts a near-random label.

This failure connects directly to **Assumption 3** from Task 1.2: the hinge-loss bound is only meaningful when there is a clear, achievable margin. With too many components (Assumption 3 violation) and too little regularisation (C1 too small), the large-margin principle collapses. It also connects to **Assumption 1** (mean-field): when samples are split thinly across many components, the mean-field assignment entropy is high and the coupling between the feature model and the classifier model breaks down.

**Concrete modification to address this failure:** Add a model-selection step that penalises the number of active components by including a term −λ × K_active in the objective (where K_active is the number of components with Σ_d ϕ_d^t > threshold), encouraging the DP to merge near-identical components before training the component SVMs.
